In [0]:
# Configuration
from pyspark.sql import functions as F, Window

dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.dropdown("run_checks", "true", ["true", "false"], "Verification / experimentation")
RUN_CHECKS = dbutils.widgets.get("run_checks") == "true"
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_SENSOR = f"{CATALOG}.{BRONZE_SCHEMA}.sensor_data"

In [0]:
def reconcile(name, source_df, source_key, source_name, target_df, target_key, target_name):
    # Cast to string - the same business key can be INT in one layer and STRING in another

    source_keys = source_df.select(F.col(source_key).cast("string").alias("k")).distinct()
    target_keys = target_df.select(F.col(target_key).cast("string").alias("k")).distinct()

    # exceptAll is set subtraction - keys present on one side but absent from the other.
    # Both directions matter. One means data loss, the other unexpected arrivals.
    missing_in_target = source_keys.exceptAll(target_keys)
    missing_in_source = target_keys.exceptAll(source_keys)

    source_n, target_n = source_keys.count(), target_keys.count()
    missing_target_n = missing_in_target.count()
    missing_source_n = missing_in_source.count()

    print(f"\n=== {name} ===")
    print(f"  source keys: {source_n:,}")
    print(f"  target keys: {target_n:,}")
    print(f"  in source but not target: {missing_target_n:,}")
    print(f"  in target but not source: {missing_source_n:,}")

    # Persist so the metric can be tracked over time and queried by an alert
    (spark.createDataFrame(
        [(name, source_name, target_name,
          source_n, target_n, missing_target_n, missing_source_n)],
        "check_name string, source_table string, target_table string, "
        "source_keys long, target_keys long, missing_in_target long, missing_in_source long")
     .withColumn("checked_at", F.lit(check_ts).cast("timestamp"))
     .write.format("delta").mode("append").saveAsTable(results_table))

    return missing_in_target, missing_in_source